# Hettich MIKRO 220 Robotic quickstart

The MIKRO 220 Robotic is a compact centrifuge with a motorized loading hatch and rotor positioning for automated loading.

| Property | Value |
|---|---|
| Communication | RS-232 through a serial or USB-to-serial adapter |
| Serial settings | 9600 baud, 7 data bits, even parity, 1 stop bit |
| Protocol generation | Hettich Generation 2 |
| Target-speed duration | 1-second resolution; acceleration and braking excluded |
| Rotor 2334 | 24 × 2.0 mL; 13,000 rpm; 18,327 × g |
| Rotor 2394 | 24 × 2.0 mL; 13,000 rpm; 18,516 × g |
| Power | 460 VA |
| Cooling | Air-cooled; no chamber-temperature API |

Specifications: [MIKRO 220 Robotic product page](https://www.hettichlab.com/products/centrifuges/automated-centrifuges/mikro-220-robotic/) and [manufacturer data sheet](https://www.hettichlab.com/downloadcenter/Products/Datasheets/MIKRO220Robotic_EN.pdf).

Follow Hettich's rotor, accessory, balancing, installation, and guarding requirements before permitting motion.

```{device-card} hettich-mikro-220-robotic
```

## How it talks

The driver sends addressed Hettich ENQUIRY frames for state and SELECT frames for commands. It verifies the address, parameter, hexadecimal value, and XOR block-check character in every reply. A rejected command automatically reads and decodes the SIOF serial-fault register.

## Physical setup

1. Install only rotor 2334 or 2394 with approved accessories, following the Hettich operating instructions. Confirm the catalog number printed on the installed rotor.
2. Keep the rotor empty for initial hatch and positioning checks. Balance every later load by mass and position; never run an incomplete or uncertain loading pattern.
3. Close and lock the **main centrifuge lid**. The motorized **loading hatch** is a separate opening within that lid.
4. Turn the key-operated switch to `LOCK 2` for remote control.
5. Connect the serial adapter and power on the centrifuge.

## Connect

`setup()` opens the serial port, clears and reads the startup SIOF state, and verifies the exact MIKRO device code, Generation 2 identifier, and firmware version. It does not move the machine.

Replace `<YOUR_SERIAL_PORT>` with the adapter path or COM port for this installation. Set `ROTOR_CATALOG_NUMBER` from the marking on the installed rotor.

In [ ]:
from pylabrobot.hettich import HettichMikro220RoboticCentrifuge, MIKRO_220_ROBOTIC_ROTORS

ROTOR_CATALOG_NUMBER = "2334"  # Change to "2394" when that rotor is installed.
centrifuge = HettichMikro220RoboticCentrifuge(
  port="<YOUR_SERIAL_PORT>",
  rotor_catalog_number=ROTOR_CATALOG_NUMBER,
)
await centrifuge.setup()

## Confirm identity

The detected model and firmware are available after setup.

In [ ]:
print("Model:", centrifuge.device_type)
print("Firmware:", centrifuge.software_version)

## Inspect the rotor table and convert RPM and RCF

Catalog numbers 2334 and 2394 identify two different 24-place rotors, not commands or error codes. Their radii produce slightly different relative centrifugal force (RCF). The selected rotor makes both conversion directions available programmatically.

In [ ]:
for catalog_number, rotor in MIKRO_220_ROBOTIC_ROTORS.items():
  print(
    catalog_number,
    f"{rotor.positions} x {rotor.maximum_volume / 1000:g} mL,",
    f"{rotor.maximum_speed:,} rpm, {rotor.maximum_rcf:,} x g",
  )

print("RCF at 5,000 rpm:", round(centrifuge.rcf_at_speed(5_000)), "x g")
print("RPM for 5,000 x g:", centrifuge.speed_for_rcf(5_000), "rpm")

## Read machine status

`request_status()` reports the spin phase, errors, current program, rotor, main-lid state, key position, and whether centrifugation can start.

In [ ]:
status = await centrifuge.request_status()
print(status)

## Read loading-hatch and positioning status

The loading hatch has its own motion and closed signals, separate from the main lid.

In [ ]:
hatch = await centrifuge.request_hatch_status()
print(hatch)

## Read actual speed

`request_speed()` returns the measured rotor speed in rpm.

In [ ]:
print("Actual speed:", await centrifuge.request_speed(), "rpm")

## Read the installed rotor's speed limit

The live limit is checked again by `spin()` before motion.

In [ ]:
print("Rotor speed limit:", await centrifuge.request_maximum_speed(), "rpm")

## Read the device run timer

`request_elapsed_time()` returns the current acceleration-inclusive device timer in seconds. `spin()` accounts for this internally so its public `duration` means time at target speed.

In [ ]:
print("Device run timer:", await centrifuge.request_elapsed_time(), "s")

## Open the loading hatch

With an empty rotor and the main lid closed, `open_hatch()` checks the current state and sends a command only when needed. It requires standstill and `LOCK 2`, then waits for the open sensor.

In [ ]:
await centrifuge.open_hatch()

## Position the empty rotor

`move_to_position()` accepts only positions reported for the installed rotor. The main lid must be closed; the smaller loading hatch may be open, closed, or opening. Use `speed="slow"` for the initial empty-rotor check and agitation-sensitive samples. The rotor remains held in positioning mode when the method returns.

In [ ]:
await centrifuge.move_to_position(1, speed="slow")

## End positioning mode

`end_positioning()` releases the positioning hold if it is active. Repeating it is safe.

In [ ]:
await centrifuge.end_positioning()

## Close the loading hatch

`close_hatch()` is state-based and waits for both loading-hatch closed signals.

In [ ]:
await centrifuge.close_hatch()

## Recall a stored program

`recall_program()` activates a program from the centrifuge's stored range 1–89. Recalling the already-active program is a no-op. The later `spin()` call replaces runtime and speed while retaining the program's other settings, such as acceleration and braking profiles.

In [ ]:
await centrifuge.recall_program(1)

## Load, balance, and verify both closures

After the empty-rotor checks, load and balance the rotor using the approved procedure. Before spinning, confirm that the main lid is locked and both loading-hatch closed signals are active.

In [ ]:
status = await centrifuge.request_status()
hatch = await centrifuge.request_hatch_status()
print("Main lid closed:", status.lid_closed)
print("Loading hatch closed:", hatch.hatch_closed)
print("Loading hatch lock closed:", hatch.lid_lock_closed)
print("Centrifugation possible:", status.can_start)

## Run a balanced test

Stop here until the rotor has a fully balanced, approved load and the protected installation is cleared for motion. This example is deliberately gentle: 500 rpm for 10 seconds. `duration` is time at target speed; acceleration and braking are excluded. The method validates the native timer range before START, checks all motion prerequisites, and returns only after standstill.

In [ ]:
await centrifuge.spin(duration=10, speed=500)

## Emergency-stop an active run

The Hettich manual classifies PC STOP as an **emergency stop**. Use `stop_spin()` only when an active cycle must be aborted. It is a no-op at standstill.

In [ ]:
await centrifuge.stop_spin()

## Disconnect

`stop()` closes the serial connection without changing the centrifuge's run state.

In [ ]:
await centrifuge.stop()